# Data pipeline with Python: ETL over an orders dataset

The exercise turns an exploratory analysis into a small ETL flow: extract a raw orders CSV, transform it (type discipline, cleaning, outlier removal, feature engineering), and load the curated result to disk. The dataset is a retail orders extract (`dataset/orders.csv`) with intentionally dirty spots: missing values, integer discount percentages, and price outliers.

In [ ]:
import pandas as pd
import numpy as np
import os

## Extract

Types are pinned at read time rather than fixed afterwards. `Postal Code` and the two id columns are forced to `str` (a numeric cast would drop leading zeros and turn identifiers into quantities), and `Order Date` is parsed directly by `read_csv`. Column names are normalized to `snake_case` immediately so every later step works against one naming convention. Numeric columns go through `pd.to_numeric(errors="coerce")`: a value that cannot be parsed becomes `NaN` now and gets handled by the cleaning step, instead of surfacing as an `object` column three cells later.

In [ ]:
dtype_rules = {
    'Postal Code': str,  # keeps leading zeros
    'Product Id': str,
    'Order Id': str
}

df = pd.read_csv('dataset/orders.csv',
                 dtype=dtype_rules,
                 parse_dates=['Order Date'])

# one naming convention for the whole pipeline: "Order Id" -> "order_id"
df.columns = df.columns.str.lower().str.replace(' ', '_')

numeric_cols = ['cost_price', 'list_price', 'discount_percent', 'quantity']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Loaded dataset, initial shape: {df.shape}")
df.head(3)

## Transform: cleaning

Rows with missing values are dropped rather than imputed. For a KPI pipeline on transactional rows this is the defensible default: imputing a price or a quantity fabricates revenue. The discount column arrives as an integer percentage (5 means 5%), so it is rescaled to a decimal fraction once, here, before any monetary computation depends on it.

In [ ]:
initial_rows = len(df)
df = df.dropna()
print(f"Rows dropped for missing values: {initial_rows - len(df)}")

# discount arrives as an integer percentage (5 -> 0.05)
df['discount_percent'] = df['discount_percent'] / 100

## Transform: outlier removal

Price and quantity outliers are filtered with the IQR rule (1.5 x IQR beyond the quartiles), applied column by column. The rule is crude but transparent: it needs no distributional assumption and it is easy to audit which rows disappeared. The cost is real too, and worth stating: sequential per-column filtering means the bounds of later columns are computed on data already trimmed by earlier ones, and legitimate bulk orders in the tail get cut together with data-entry errors.

In [ ]:
def remove_outliers(df, columns):
    df_clean = df.copy()
    rows_before = len(df_clean)

    for col in columns:
        q1 = df_clean[col].quantile(0.25)
        q3 = df_clean[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]

    print(f"Outliers removed: {rows_before - len(df_clean)} rows")
    return df_clean

cols_check = ['list_price', 'cost_price', 'quantity']
df = remove_outliers(df, cols_check)

print(f"Post-outlier shape: {df.shape[0]} rows")

## Transform: feature engineering

Two families of derived columns. Economic KPIs (discounted unit price, revenue, cost, profit, margin) are computed row-wise from the cleaned base columns; the margin uses `np.where` to guard the division when revenue is zero. Temporal features (year, month, day of week, weekend flag) are extracted from the parsed order date and are the kind of columns a downstream model or BI layer expects to find ready-made.

In [ ]:
# economic KPIs
df['sale_price_unit'] = df['list_price'] * (1 - df['discount_percent'])
df['sales_amount'] = df['sale_price_unit'] * df['quantity']
df['total_cost'] = df['cost_price'] * df['quantity']
df['profit'] = df['sales_amount'] - df['total_cost']
df['profit_margin'] = np.where(df['sales_amount'] > 0,
                               (df['profit'] / df['sales_amount']) * 100,
                               0)

# temporal features
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['order_day_of_week'] = df['order_date'].dt.dayofweek  # 0=Mon, 6=Sun
df['is_weekend'] = df['order_day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

df[['order_id', 'sales_amount', 'profit', 'order_month']].head()

## Load

The curated dataset lands as a flat CSV next to the notebook. CSV is the simplest handoff format for the exercise; the trade-offs against Parquet are picked up below.

In [ ]:
output_file = 'clean_orders.csv'

df.to_csv(output_file, index=False, encoding='utf-8')
size_kb = os.path.getsize(output_file) / 1024

print(f"Saved {output_file} ({size_kb:.2f} KB, {len(df)} rows)")

## Critical notes and follow-ups

The notebook is the exploratory form of the pipeline, not its production form. Three upgrades would move it toward the target the lesson outline describes:

- Modularization: fold the cells into `extract_data()`, `transform_data()`, `load_data()` functions so each stage is testable in isolation and the flow is re-runnable end to end.
- Logging: replace `print` with the `logging` module (INFO for stage progress, ERROR for failures) so a scheduled run leaves an inspectable trace.
- Persistence: swap the CSV for Parquet (typed, compressed, columnar) or a local PostgreSQL table when a downstream consumer needs concurrent or partial reads.

On data quality, the two lossy choices (drop instead of impute, IQR trimming) are fine for a reporting pipeline but would need revisiting if the output fed a model: dropped rows are silent training-distribution shifts, and the per-column IQR pass has no notion of a legitimate heavy tail.